# Certified defenses

This code trains a neural network on a simple binary classification task while using Interval Bound Propagation (IBP) to improve adversarial robustness. IBP ensures that the model is not only accurate on clean data but also robust to small perturbations in inputs.

We import our required packages and creates a synthetic dataset of two-dimensional inputs (x values). This code begins by importing TensorFlow, Keras, and NumPy, then defines a function to generate a simple synthetic dataset for a binary classification task. The dataset consists of random two-dimensional input features and corresponding binary labels, where the rule is straightforward: if the sum of the two features exceeds one, the label is 1; otherwise, it is 0. To ensure reproducibility, a random seed is set, and NumPy generates the input array from a uniform distribution between 0 and 1. Labels are created by summing the two feature columns, applying the threshold, and converting the boolean result into integers. Finally, the function is used to generate 1,000 training samples and 200 test samples.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Step 1: Generate simple data
def generate_data(num_samples=1000):
    np.random.seed(42)
    x = np.random.rand(num_samples, 2)  # Two-dimensional input
    y = (x[:, 0] + x[:, 1] > 1).astype(int)  # Simple binary classification
    return x, y

x_train, y_train = generate_data()
x_test, y_test = generate_data(200)

We do our preprocessing steps. This step converts the NumPy arrays created earlier into TensorFlow tensors, the core data structure used for computation in TensorFlow. Both the training and testing features are cast to float32, a standard format for neural network inputs, while the labels are also converted to float32 to support loss calculations. Since the model will output a single value per sample in this binary classification task, the labels are reshaped into column vectors (e.g., from (1000,) to (1000, 1)), ensuring their dimensions align with the expected output shape of the model.

In [ ]:
# Convert to TensorFlow tensors and ensure proper shapes
x_train = tf.convert_to_tensor(x_train, dtype=tf.float32)
x_test = tf.convert_to_tensor(x_test, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)

# Reshape labels to match the model's output shape
y_train = tf.reshape(y_train, (-1, 1))
y_test = tf.reshape(y_test, (-1, 1))

We define a simple model to test. This function defines a simple sequential neural network for binary classification using fully connected (dense) layers. The model begins with a dense layer that maps the two input features into 16 dimensions using the ReLU activation function, which passes positive values through unchanged and outputs zero otherwise. A second dense layer further processes the data with another 16 ReLU-activated units. Finally, the output layer reduces the representation to a single value with a sigmoid activation, which compresses outputs to a range between 0 and 1, making them interpretable as probabilities for binary classification.

In [ ]:
# Define a simple model
def create_model():
    model = models.Sequential([
        layers.Dense(16, activation='relu', input_shape=(2,)),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

This function implements the Interval Bound Propagation (IBP) loss, which combines accuracy on clean data with robustness to small input perturbations. It first computes the standard binary cross-entropy loss between the model’s predictions and the true labels on the original inputs. To account for adversarial variation, it then defines an interval around each input by subtracting and adding a small value (ε), clipped to remain within the valid range. The model’s predictions at the lower and upper bounds of this interval are evaluated, and the worst-case binary cross-entropy loss is taken as the robust loss. The final IBP loss is the sum of the clean loss and robust loss, encouraging the model to perform well on unmodified data while also resisting adversarial perturbations.

In [ ]:
# Interval Bound Propagation (IBP) loss function
def ibp_loss(model, x, y, epsilon=0.1):
    """
    Computes the IBP loss, combining clean loss and worst-case robust loss.
    :param model: The neural network model
    :param x: Input data
    :param y: True labels
    :param epsilon: Perturbation range
    :return: Total loss
    """
    with tf.GradientTape() as tape:
        tape.watch(x)
        logits = model(x, training=True)
        y = tf.cast(y, tf.float32)  # Ensure labels are float32
        clean_loss = tf.keras.losses.binary_crossentropy(y, logits)

        # Compute bounds
        lower_bound = tf.clip_by_value(x - epsilon, 0.0, 1.0)
        upper_bound = tf.clip_by_value(x + epsilon, 0.0, 1.0)
        worst_logits_lower = model(lower_bound)
        worst_logits_upper = model(upper_bound)
        robust_loss = tf.maximum(
            tf.keras.losses.binary_crossentropy(y, worst_logits_lower),
            tf.keras.losses.binary_crossentropy(y, worst_logits_upper)
        )
    return clean_loss + robust_loss

This section demonstrates how to train and evaluate a model using Interval Bound Propagation (IBP). The training function minimises the IBP loss with the Adam optimiser over several epochs, iterating through the training data in batches. For each batch, the IBP loss is calculated, gradients are derived, and model weights are updated accordingly, with the loss reported at the end of each epoch. After training, two evaluation functions are provided. The first measures standard accuracy on clean, unperturbed test data by thresholding predictions at 0.5 and comparing them with true labels. The second evaluates robust accuracy by introducing perturbations within a defined epsilon range. It checks predictions at both the lower and upper bounds of this interval and counts a sample as correct only if both match the true label. Together, these steps allow us to compare how well the model performs under normal conditions versus adversarially perturbed inputs, highlighting the practical impact of training with IBP.

In [ ]:
# Train the model with IBP
def train_with_ibp(model, x_train, y_train, epochs=5, batch_size=32, epsilon=0.1):
    """
    Trains the model using Interval Bound Propagation (IBP).
    :param model: The neural network model
    :param x_train: Training data
    :param y_train: Training labels
    :param epochs: Number of training epochs
    :param batch_size: Batch size
    :param epsilon: Perturbation range
    """
    optimizer = tf.keras.optimizers.Adam()
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        for i in range(0, len(x_train), batch_size):
            x_batch = x_train[i:i + batch_size]
            y_batch = y_train[i:i + batch_size]

            with tf.GradientTape() as tape:
                loss = ibp_loss(model, x_batch, y_batch, epsilon=epsilon)
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        print(f"Loss at end of epoch: {loss.numpy()}")

# Evaluate the model
def evaluate_model(model, x_test, y_test):
    """
    Evaluates the model on clean test data.
    :param model: The trained model
    :param x_test: Test data
    :param y_test: Test labels
    :return: Accuracy
    """
    predictions = (model.predict(x_test) > 0.5).astype(int).flatten()
    true_labels = y_test.numpy().flatten()
    accuracy = np.mean(predictions == true_labels)
    return accuracy

def evaluate_with_perturbations(model, x_test, y_test, epsilon=0.1):
    """
    Evaluates the model's robustness under adversarial perturbations.
    :param model: The trained model
    :param x_test: Test data
    :param y_test: Test labels
    :param epsilon: Perturbation range
    :return: Accuracy
    """
    lower_bound = tf.clip_by_value(x_test - epsilon, 0.0, 1.0)
    upper_bound = tf.clip_by_value(x_test + epsilon, 0.0, 1.0)

    # Compute worst-case predictions
    lower_predictions = (model.predict(lower_bound) > 0.5).astype(int).flatten()
    upper_predictions = (model.predict(upper_bound) > 0.5).astype(int).flatten()
    true_labels = y_test.numpy().flatten()

    # If either bound predicts incorrectly, mark as misclassified
    robust_predictions = (lower_predictions == true_labels) & (upper_predictions == true_labels)
    accuracy = np.mean(robust_predictions)
    return accuracy

# Train and evaluate the model
model = create_model()



Training with IBP...
Epoch 1/5
Loss at end of epoch: [1.4006355 1.1278156 1.1200504 1.5482621 1.513531  1.0547117 1.30269
 1.4351315]
Epoch 2/5
Loss at end of epoch: [1.3305402 1.0202222 1.0908272 1.6827204 1.6015872 0.9335353 1.2033393
 1.333371 ]
Epoch 3/5
Loss at end of epoch: [1.2161114 0.9489577 1.1093674 1.7857263 1.6563423 0.8398111 1.1376691
 1.1864471]
Epoch 4/5
Loss at end of epoch: [1.0194259  0.8852056  1.1541404  1.879138   1.6751125  0.7501018
 1.0913541  0.97269607]
Epoch 5/5
Loss at end of epoch: [0.827609   0.8153999  1.188961   1.9937125  1.6517398  0.6405363
 1.0340991  0.79183435]
Evaluating on clean test data...
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Original Accuracy: 86.50%
Evaluating on perturbed test data...
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Robust Accuracy (with perturbations): 68.00%


The final code block runs the training and evaluation process. It begins by printing a message and calling the train_with_ibp function to train the model for five epochs with an epsilon of 0.1. Once training is complete, the model is evaluated on clean test data using evaluate_model, and the resulting accuracy is printed as a percentage. To assess robustness, the model is then tested on perturbed data within the same epsilon range using evaluate_with_perturbations. The robust accuracy, representing performance under adversarial perturbations, is also printed, allowing a direct comparison between the model’s standard and adversarial resilience.

In [ ]:
print("Training with IBP...")
train_with_ibp(model, x_train, y_train, epochs=5, epsilon=0.1)

print("Evaluating on clean test data...")
original_accuracy = evaluate_model(model, x_test, y_test)
print(f"Original Accuracy: {original_accuracy * 100:.2f}%")

print("Evaluating on perturbed test data...")
robust_accuracy = evaluate_with_perturbations(model, x_test, y_test, epsilon=0.1)
print(f"Robust Accuracy (with perturbations): {robust_accuracy * 100:.2f}%")